In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [3]:
# Load dataset
boston = fetch_openml(name="boston", version=1, as_frame=False)
X = boston.data.astype(np.float32)
y = boston.target.astype(np.float32).reshape(-1, 1)

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features (important for NN training)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
X_train

array([[ 1.287702  , -0.50032014,  1.0332369 , ...,  0.8453428 ,
        -0.07433686,  1.753505  ],
       [-0.3363845 , -0.50032014, -0.41315946, ...,  1.204741  ,
         0.43018365, -0.5614742 ],
       [-0.40325332,  1.0132713 , -0.7152182 , ..., -0.6371767 ,
         0.06529737, -0.6515951 ],
       ...,
       [-0.40547013,  2.9593174 , -1.3033613 , ..., -0.59225166,
         0.37900996, -0.9106926 ],
       [ 0.8518974 , -0.50032014,  1.0332369 , ...,  0.8453428 ,
        -2.694586  ,  1.5225705 ],
       [-0.38135594, -0.50032014, -0.35216683, ...,  1.159816  ,
        -3.1215804 , -0.25731638]], shape=(404, 13), dtype=float32)

## Boston Dataset Features

| Feature | Description |
|---------|-------------|
| CRIM | Per capita crime rate by town |
| ZN | Proportion of residential land zoned for large lots |
| INDUS | Proportion of non-retail business acres per town |
| NOX | Nitric oxides concentration |
| RM | Average number of rooms per dwelling |
...

**Target:** median house value in $1000s.

In [5]:
# convert to tensors
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train)
X_test = torch.tensor(X_test)
y_test = torch.tensor(y_test)

In [18]:
# Neural network model (regression)
class HousingNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)   # no sigmoid → regression output
        )

    def forward(self, x):
        return self.net(x)

model = HousingNet(X_train.shape[1])

In [19]:
# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [20]:
# Training loop
epochs = 1000
for epoch in range(epochs):

    # forward
    preds = model(X_train)
    loss = criterion(preds, y_train)

    # backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch:3d} | Train MSE: {loss.item():.4f}")

Epoch   0 | Train MSE: 605.5602
Epoch 100 | Train MSE: 72.6316
Epoch 200 | Train MSE: 23.8841
Epoch 300 | Train MSE: 17.3578
Epoch 400 | Train MSE: 13.7511
Epoch 500 | Train MSE: 11.5948
Epoch 600 | Train MSE: 10.4343
Epoch 700 | Train MSE: 9.5426
Epoch 800 | Train MSE: 8.7467
Epoch 900 | Train MSE: 7.9669


In [21]:
# Evaluation

with torch.no_grad():
    test_preds = model(X_test)
    test_mse = criterion(test_preds, y_test)
    rmse = torch.sqrt(test_mse)

print("\nTest RMSE:", rmse.item())


Test RMSE: 3.3024463653564453


target values range roughly from $5k to $50k, so ~$3.3k error is a reasonable result for a simple MLP